## Configurações

In [1]:
# Extensão que verifica se um dos arquivos foi alterado
%load_ext autoreload
%autoreload 2

In [13]:
from c3_1_instance_reweighing import instance_reweighing  # É preciso começar por ele para não dar problema com o torch vindo da aif360

# Bibliotecas
import pandas as pd
import joblib
import traceback
import os


# Variáveis auxiliares
from c0_1_configuracoes import(
  param_grid_perceptron_basico,
  param_grid_random_forest_basico,
  param_grid_regressao_logistica_basico,
  param_grid_xgboost_basico,
  param_grid_perceptron_completo,
  param_grid_random_forest_completo,
  param_grid_regressao_logistica_completo,
  param_grid_xgboost_completo,
  preprocessor_passthrough
)

from c0_2_cronometro import cronometro

# Funções auxiliares
from c1_6_enviesamento import enviesar
from c1_7_salvar_resultados import gerar_planilha, salvar_dicionario

# Algoritmos
from c2_1_random_forest import random_forest_GSCV
from c2_2_xgboost import xgboost_GSCV
from c2_3_regressao_logistica import regressao_logistica_GSCV
from c2_4_perceptron import perceptron_GSCV

# Técnicas de pré-processamento
from c3_1_instance_reweighing import instance_reweighing
from c3_2_disparate_impact_removal import disparate_impact_removal
from c3_3_synthetic_data_generation import synthetic_data_generation
from c3_4_suppression import suppression

# Técnicas de pós-processamento
from c4_1_threshold_optimization import threshold_optimization
from c4_2_calibration import calibration
from c4_3_reject_option_classification import reject_option_classification

caminho_resultado = './Resultados'

## Cálculo

In [ ]:
# Chama a função que aplica todos os enviesamento considerando as variáveis sensiveis
# Não compensa salvar o arquivo pois seria muito pesado e leva apenas ~10 segundos para executar
datasets = enviesar()

In [ ]:
modo_completo = 1

param_grid_random_forest = param_grid_random_forest_completo if modo_completo else param_grid_random_forest_basico
param_grid_perceptron = param_grid_perceptron_completo if modo_completo else param_grid_perceptron_basico
param_grid_regressao_logistica = param_grid_regressao_logistica_completo if modo_completo else param_grid_regressao_logistica_basico
param_grid_xgboost = param_grid_xgboost_completo if modo_completo else param_grid_xgboost_basico

algoritmos = {
    'random_forest': {
        'nome_modelo': 'RANDOM FOREST',
        'funcao': random_forest_GSCV,
        'parametros': param_grid_random_forest
    },
    'perceptron': {
        'nome_modelo': 'PERCEPTRON',
        'funcao': perceptron_GSCV,
        'parametros': param_grid_perceptron
    },
    'regressão_logística': {
        'nome_modelo': 'REGRESSÃO LOGÍSTICA',
        'funcao': regressao_logistica_GSCV,
        'parametros': param_grid_regressao_logistica
    },
    'xgboost': {
        'nome_modelo': 'XGBOOST',
        'funcao': xgboost_GSCV,
        'parametros': param_grid_xgboost
    }
}

printar_tecnicas = False

# Parâmetros comuns para todos
parametros = {}
parametros['printar'] = True
parametros['cv_n_splits'] = 2

resultado_global = {}
sucesso = 0

# Percorre cada base de dados distinta
try:

  for df in datasets:

    print(f"\n\n=== INICIANDO ANÁLISE DO {df.upper()} ===\n")
    # Percorre cada tipo de enviesamento
    for tipo in datasets[df]:

      print(f"\n--- Analisando o tipo: {tipo} ---\n")

      banco = datasets[df][tipo]
      nome_banco = banco['nome_banco']

      # Separando dados de treino e teste
      X_train = banco['treino'].drop('target',axis=1)
      y_train = banco['treino']['target']

      X_test = banco['teste'].drop('target', axis=1)
      y_test = banco['teste']['target']

      # Passando os dados em dataframes já separados
      parametros_de_treino = parametros
      parametros_de_treino['X_train'] = X_train
      parametros_de_treino['X_test'] = X_test
      parametros_de_treino['y_train'] = y_train
      parametros_de_treino['y_test'] = y_test
      parametros_de_treino['preprocessor'] = preprocessor_passthrough # Já foram pré-processados antes
      parametros_de_treino['dados_sensiveis'] = banco['dados_sensiveis']

      resultado_dataset = {}
      resultado_dataset['smote'] = banco['smote']

      for algoritmo in algoritmos:
        print(f"\n### Analisando o algoritmo: {algoritmo.upper()} ###\n")

        algoritmo = algoritmos[algoritmo]

        parametros_de_treino['param_grid'] = algoritmo['parametros']
        parametros_de_treino['nome_base_de_dados'] = nome_banco + f" || {algoritmo['nome_modelo']}"

        desempenho = {}

        # Pré-processamento
        with cronometro() as timer:
          desempenho['instance_reweighing']               = instance_reweighing(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['instance_reweighing']['tempo'] = timer()

        with cronometro() as timer:
          desempenho['disparate_impact_removal']     = disparate_impact_removal(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['disparate_impact_removal']['tempo'] = timer()

        with cronometro() as timer:
          desempenho['synthetic_data_generation']   = synthetic_data_generation(algoritmo['funcao'], parametros_de_treino, colunas_discretas=banco['colunas_discretas'], printar=printar_tecnicas)
        desempenho['synthetic_data_generation']['tempo'] = timer()

        with cronometro() as timer:  
          desempenho['suppression']                               = suppression(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['suppression']['tempo'] = timer()

        # Pós-processamento
        with cronometro() as timer: 
          (_, desempenho['threshold_optimization'])                         = threshold_optimization(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['threshold_optimization']['tempo'] = timer()

        with cronometro() as timer: 
          (_, desempenho['calibration'])                                               = calibration(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['calibration']['tempo'] = timer()

        with cronometro() as timer: 
          (_, desempenho['reject_option_classification'])             = reject_option_classification(algoritmo['funcao'], parametros_de_treino, printar=printar_tecnicas)
        desempenho['reject_option_classification']['tempo'] = timer()

        # Desempenho original (sem técnica)
        parametros_de_treino['nome_base_de_dados'] = parametros_de_treino['nome_base_de_dados'] + " || SEM TÉCNICA"
        with cronometro() as timer:
          (_, desempenho['sem_tecnica']) = algoritmo['funcao'](**parametros_de_treino)
        desempenho['sem_tecnica']['tempo'] = timer()

        resultado_dataset[f"{algoritmo['nome_modelo'].lower().replace(' ', '_')}"] = desempenho

      resultado_global[f"{nome_banco.lower().replace(' ', '_')}"] = resultado_dataset.copy()

except KeyboardInterrupt:
  traceback.print_exc()


except Exception:
  traceback.print_exc()

else:
  sucesso = 1

finally:
  print("\n\n\n-------------------------------------------")
  salvar_dicionario(resultado_global, caminho_resultado, sucesso)
  gerar_planilha(resultado_global, caminho_resultado)
  print("-------------------------------------------\n\n\n")



=== INICIANDO ANÁLISE DO DF1 ===


--- Analisando o tipo: original_sensitive_sexo ---


### Analisando o algoritmo: RANDOM_FOREST ###




-------------------------------------------
Atenção! Há um dicionário com o mesmo nome!
Salvando com o nome _aux
Dicionário com o resultado parcial salvo em ./Resultados/resultado_parcial_dict_aux_12_12_2025_16_28.joblib
Atenção! Há uma planilha com o mesmo nome!
Salvando com o nome _aux
Arquivo Excel com 0 linhas de dados salvo em ./Resultados/resultado_global_aux_12_12_2025_16_28.xlsx
-------------------------------------------





Traceback (most recent call last):
  File "C:\Users\Gabriel\AppData\Local\Temp\ipykernel_2884\1580032480.py", line 117, in <module>
    (_, desempenho['sem_tecnica']) = algoritmo['funcao'](**parametros_de_treino)
                                     ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Gabriel\Documents\IC\Codigo\c2_1_random_forest.py", line 30, in random_forest_GSCV
    grid_search.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\model_selection\_search.py", line 1023, in fit
    self._run_search(evaluate_candidates)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Gabriel\Documents\IC\venv\Lib\site-packages\sklearn\model_selection\_search.py", line 1570, in _run_search
    evaluate_candidates(Par

## Gerar arquivo tabela manualmente

In [20]:
dicionario = './Resultados/resultado_parcial_dict_12_12_2025_16_15.joblib'

if os.path.isfile(dicionario):
  rf = joblib.load(dicionario)
  print("Dicionário recuperado")

Dicionário recuperado


In [37]:
gerar_planilha(rf, caminho_resultado)

Arquivo Excel com 0 linhas de dados salvo em ./Resultados/resultado_global_12_12_2025_16_31.xlsx


## Visualizar o dicionário de resultado

In [36]:
def listar_hierarquia_ordenada(dicionario, nivel=0):
    # Ordenamos as chaves para garantir que a lista fique organizada (alfabeticamente)
    for chave, valor in sorted(dicionario.items()):
        # Cria 4 espaços para cada nível de profundidade
        indentacao = "    " * nivel
        print(f"{indentacao}{chave}")
        
        # Se o valor for um dicionário, chama a função novamente (recursão)
        if isinstance(valor, dict):
            listar_hierarquia_ordenada(valor, nivel + 1)